# FunnelLens: mobile checkout, or something else?

The Python track's report. Every number below is computed by the `funnellens` package from the raw export, so re-running this notebook from a clean checkout reproduces it exactly (CI executes it with `make report`). The formulas are in `contract/questions.md`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from funnellens.pipeline import analyse

DATA = Path(os.environ.get("FUNNELLENS_DATA", "../../../data/raw/2026-08"))  # relative to notebooks/
a = analyse(DATA)
print(a["dataset"])

## Q1 · Can we trust the export?
What cleaning removed, and the contract checks (an `error` check would have stopped the run before this cell).

In [ ]:
issues = pd.Series(a["data_quality"]["issues"], name="rows")
rows = pd.DataFrame({"raw": a["data_quality"]["raw_rows"], "clean": a["data_quality"]["clean_rows"]})
display(rows)
display(issues.to_frame())
pd.DataFrame(a["data_quality"]["checks"])[["id", "severity", "status", "failing_rows", "observed"]]

## Q2 · The funnel by device

In [ ]:
by = a["funnel"]["by_device"]
funnel = pd.DataFrame({d: {s: by[d][s] for s in a["funnel"]["steps"]} for d in ("desktop", "mobile")})
display(funnel)
rates = pd.DataFrame({d: by[d]["step_rates"] for d in ("desktop", "mobile")})
ax = rates.plot.bar(figsize=(7, 3.5), color=["#9aa5b1", "#d64545"])
ax.set_title("Mobile falls behind at the cart and payment steps")
ax.set_ylabel("share of the previous step")
ax.set_xlabel("step")
plt.xticks(rotation=0)
plt.show()
print(
    f"mobile: {a['funnel']['mobile_share_of_sessions']:.1%} of sessions, "
    f"{a['funnel']['mobile_share_of_revenue']:.1%} of revenue"
)

## Q3 · Is the mobile payment step worse?
Two-proportion z test on purchase given checkout.

In [ ]:
p = a["payment_step_test"]
print(
    f"mobile {p['mobile']['rate']:.1%} vs desktop {p['desktop']['rate']:.1%}: "
    f"{p['difference'] * 100:+.1f} pp, 95 % CI [{p['ci95'][0] * 100:.1f}, {p['ci95'][1] * 100:.1f}] pp, "
    f"z = {p['z']:.2f}, p = {p['p_value']:.1e}"
)

## Q4 · Compare like with like: within each acquisition channel
The crude comparison mixes channels: mobile traffic is mostly paid social, which converts worse on every device. The Cochran-Mantel-Haenszel test compares mobile with desktop inside each channel.

In [ ]:
rows = []
for c in a["by_channel"]["channels"]:
    for d in ("desktop", "mobile"):
        n = c[d]["checkout"]
        rows.append(
            {
                "channel": c["channel"],
                "device": d,
                "checkouts": n,
                "pay_rate": c[d]["purchase"] / n if n else None,
                "cart_rate": c[d]["cart"] / c[d]["view"] if c[d]["view"] else None,
            }
        )
ch = pd.DataFrame(rows)
display(ch)
g = sns.catplot(
    data=ch, x="channel", y="pay_rate", hue="device", kind="bar", height=3.5, aspect=1.8, palette=["#9aa5b1", "#d64545"]
)
g.ax.set_title("Within every channel, mobile pays less often after checkout")
plt.show()
pd.DataFrame(a["by_channel"]["cmh"]).T

In [ ]:
s = a["by_channel"]["simpson"]
print(
    f"cart step, aggregate mobile - desktop: {s['aggregate_gap'] * 100:+.1f} pp; within channels:",
    {k: round(v * 100, 1) for k, v in s["within_channel_gaps"].items()},
    "reversal:",
    s["reversal"],
)

## Q5 · Do mobile buyers spend less?

In [ ]:
v = a["order_value"]
print(
    f"mobile ₹{v['mobile']['mean_inr']:,.0f} vs desktop ₹{v['desktop']['mean_inr']:,.0f} per order: "
    f"Welch t = {v['welch_t']:.2f}, df = {v['df']:.0f}, p = {v['p_value']:.2f}"
)

## Q6 · Is the weekly trend stable?

In [ ]:
w = pd.DataFrame(
    [
        {"week": r["week"], "device": d, "conversion": r["conversion"][d], "sessions": r["sessions"][d]}
        for r in a["weekly"]
        for d in ("desktop", "mobile")
    ]
)
fig, ax = plt.subplots(figsize=(9, 3.2))
sns.lineplot(data=w, x="week", y="conversion", hue="device", ax=ax, palette=["#9aa5b1", "#d64545"])
ax.set_title("Conversion per ISO week: no break or backfill")
ax.tick_params(axis="x", rotation=90)
plt.show()

## Q7 · What is the gap worth?
If mobile paid like desktop within each channel (the export is a 1-in-20 sample, so counts are scaled up).

In [ ]:
o = a["opportunity"]
print(
    f"₹{o['monthly_revenue_uplift_inr'] / 1e5:.1f} lakh per month "
    f"({o['uplift_share_of_mobile_revenue']:.1%} of mobile revenue)"
)
pd.DataFrame(o["by_channel"])

## Q8 · Recommendation

In [ ]:
d = a["decision"]
print(d["recommendation"], d["reasons"], d["thresholds"])

**Caveats.** Observational data: channel is the confounder we can see, not the only one. The uplift assumes mobile could reach desktop's within-channel payment rate, which only an experiment can show.

## v1.1 · Data-quality monitoring
Full ISO weeks compared with the median week. An excluded week (`cleaning.exclude_weeks` in the contract) is listed with its reason in the post-mortem.

In [ ]:
m = a["monitoring"]
print(m["freshness"], "excluded:", m["excluded_weeks"])
w = pd.DataFrame(m["weeks"]).set_index("week")
ax = w["mobile_checkout_given_cart"].plot(figsize=(9, 3), marker="o", color="#d64545")
ax.axhline(m["medians"]["mobile_checkout_given_cart"], ls="--", color="#9aa5b1")
ax.set_title("Mobile checkout given cart per full week (median dashed)")
plt.show()
pd.DataFrame(m["anomalies"])

## v1.1 · Payment methods (exports from 2026-08)

In [ ]:
pm = a["payment_methods"]
pd.DataFrame(pm["by_device"]).fillna(0).astype(int) if pm else "not in this export"